# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ManjusreeValluri/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [14]:
!git clone https://github.com/ManjusreeValluri/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [15]:
!ls /content/flyrank-ml-internship/data/raw

content_refresh_anonymized.csv


In [16]:
import pandas as pd

data_path = "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

Shape: (30000, 44)
Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [17]:
# Signal 1: staleness
print(
    df.groupby("freshness_tier")
      .agg(
          n=("content_id", "size"),
          avg_ctr=("ctr", "mean")
      )
      .reset_index()
)

  freshness_tier      n   avg_ctr
0           0-30  20480  0.609021
1           181+    174  3.693276
2          31-90    175  0.117543
3         91-180   9171  0.238367


In [18]:
# Signal 2: CTR relative to search position
print(
    df.groupby("position_tier")
      .agg(
          n=("content_id", "size"),
          avg_ctr=("ctr", "mean")
      )
      .reset_index()
)

  position_tier      n   avg_ctr
0          deep   1319  0.150212
1        page_1  11814  0.652467
2      page_3_5   7242  0.222484
3      striking   7304  0.323239
4         top_3   2321  1.483611


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Signal checks

**Signal 1 — Freshness / staleness:**  
I checked average CTR across freshness tiers. The results are different across the buckets, but the pattern is not consistently worse for older content. The 181+ group has the highest average CTR, so I treat this signal as **MIXED** rather than assuming that old content always needs a refresh.

**Signal 2 — Search position:**  
I checked average CTR across position tiers. CTR is much higher for pages in the top 3 and page 1 than for deeper positions. I treat this signal as **CONFIRMED** because the measured pattern supports the expected relationship between search position and CTR.

### Rule

Prioritize pages where the search position suggests an opportunity for CTR improvement, while using content freshness as supporting context rather than assuming that older content is automatically worse.

### Reason codes

- `CTR_FIX` — the page's search position suggests an opportunity to improve CTR.
- `REFRESH_SUPPORT` — the page is relatively stale and freshness supports considering an update.

In [19]:
# Signal 1: Freshness / staleness check
freshness_check = (
    df.groupby("freshness_tier")
      .agg(
          n=("content_id", "size"),
          avg_ctr=("ctr", "mean")
      )
      .reset_index()
)

print("Signal 1 — Freshness / staleness")
print(freshness_check)


# Signal 2: Search position and CTR check
position_check = (
    df.groupby("position_tier")
      .agg(
          n=("content_id", "size"),
          avg_ctr=("ctr", "mean")
      )
      .reset_index()
)

print("\nSignal 2 — Search position and CTR")
print(position_check)

Signal 1 — Freshness / staleness
  freshness_tier      n   avg_ctr
0           0-30  20480  0.609021
1           181+    174  3.693276
2          31-90    175  0.117543
3         91-180   9171  0.238367

Signal 2 — Search position and CTR
  position_tier      n   avg_ctr
0          deep   1319  0.150212
1        page_1  11814  0.652467
2      page_3_5   7242  0.222484
3      striking   7304  0.323239
4         top_3   2321  1.483611


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Baseline scoring rule

I score pages mainly by their search-position opportunity, with freshness used as supporting context.

Pages with lower search positions receive higher CTR-fix priority. Older content can receive additional support for a refresh, but age alone is not treated as proof that a page needs updating.

Each row receives one reason code and one action label.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import os

# Start with a copy of the dataset
queue = df.copy()

# Score based mainly on search position
queue["score"] = 0.0

# Higher priority for pages with weaker search positions
queue.loc[queue["position_tier"] == "deep", "score"] += 4
queue.loc[queue["position_tier"] == "page_3_5", "score"] += 3
queue.loc[queue["position_tier"] == "striking", "score"] += 2
queue.loc[queue["position_tier"] == "page_1", "score"] += 1
queue.loc[queue["position_tier"] == "top_3", "score"] += 0

# Freshness is supporting context only
queue.loc[queue["freshness_tier"].isin(["91-180", "181+"]), "score"] += 1

# One reason code and one action
queue["reason_code"] = np.where(
    queue["position_tier"].isin(["deep", "page_3_5", "striking"]),
    "CTR_FIX",
    "REFRESH_SUPPORT"
)

queue["action"] = np.where(
    queue["reason_code"] == "CTR_FIX",
    "Improve CTR",
    "Review freshness"
)

# Rank highest score first
queue = queue.sort_values(
    ["score", "content_id"],
    ascending=[False, True]
).reset_index(drop=True)

queue["rank"] = queue.index + 1

# Keep the important columns
baseline_queue = queue[
    ["rank", "content_id", "score", "reason_code", "action"]
].copy()

# Write the required CSV
output_path = "/content/flyrank-ml-internship/work/outputs/baseline_action_score.csv"

os.makedirs(os.path.dirname(output_path), exist_ok=True)

baseline_queue.to_csv(output_path, index=False)

print("Rows:", len(baseline_queue))
print("CSV written to:", output_path)
print("\nTop 10:")
print(baseline_queue.head(10))


Rows: 30000
CSV written to: /content/flyrank-ml-internship/work/outputs/baseline_action_score.csv

Top 10:
   rank            content_id  score reason_code       action
0     1  content_00bd50f6286d    5.0     CTR_FIX  Improve CTR
1     2  content_01e6f180a032    5.0     CTR_FIX  Improve CTR
2     3  content_0239cfc771ee    5.0     CTR_FIX  Improve CTR
3     4  content_026300c5cc1e    5.0     CTR_FIX  Improve CTR
4     5  content_0334348fc2d9    5.0     CTR_FIX  Improve CTR
5     6  content_0371fc7cdb0a    5.0     CTR_FIX  Improve CTR
6     7  content_03b7a0f49f84    5.0     CTR_FIX  Improve CTR
7     8  content_04e97796ac4d    5.0     CTR_FIX  Improve CTR
8     9  content_0527b708fdec    5.0     CTR_FIX  Improve CTR
9    10  content_07445a84d693    5.0     CTR_FIX  Improve CTR


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show the top 20 rows with the signals used by the rule
top20 = queue.head(20)[
    [
        "rank",
        "content_id",
        "score",
        "reason_code",
        "action",
        "position_tier",
        "freshness_tier",
        "ctr",
        "avg_position"
    ]
].copy()

print(top20.to_string(index=False))


 rank           content_id  score reason_code      action position_tier freshness_tier  ctr  avg_position
    1 content_00bd50f6286d    5.0     CTR_FIX Improve CTR          deep         91-180 0.00          61.4
    2 content_01e6f180a032    5.0     CTR_FIX Improve CTR          deep         91-180 0.00          62.2
    3 content_0239cfc771ee    5.0     CTR_FIX Improve CTR          deep         91-180 0.00          52.7
    4 content_026300c5cc1e    5.0     CTR_FIX Improve CTR          deep         91-180 0.00          55.0
    5 content_0334348fc2d9    5.0     CTR_FIX Improve CTR          deep         91-180 0.29          54.6
    6 content_0371fc7cdb0a    5.0     CTR_FIX Improve CTR          deep         91-180 0.04          69.7
    7 content_03b7a0f49f84    5.0     CTR_FIX Improve CTR          deep         91-180 0.00          66.5
    8 content_04e97796ac4d    5.0     CTR_FIX Improve CTR          deep         91-180 0.11          51.4
    9 content_0527b708fdec    5.0     CTR_FIX 

### Top-20 review

1. **content_00bd50f6286d** — Action: Improve CTR. Reason: CTR_FIX because the page is deep in search results with CTR 0.00. Confidence: moderate. It could be wrong if the page is not realistically competitive for the target query.

2. **content_01e6f180a032** — Action: Improve CTR. Reason: CTR_FIX because the page is deep with CTR 0.00. Confidence: moderate. It could be wrong if the search position is caused by factors that a title or snippet change cannot fix.

3. **content_0239cfc771ee** — Action: Improve CTR. Reason: CTR_FIX because the page is deep with CTR 0.00. Confidence: moderate. It could be wrong if the page has little search demand or poor relevance.

4. **content_026300c5cc1e** — Action: Improve CTR. Reason: CTR_FIX because the page is deep with CTR 0.00. Confidence: moderate. It could be wrong if changing the snippet would not improve its ranking or visibility.

5. **content_0334348fc2d9** — Action: Improve CTR. Reason: CTR_FIX because the page is deep with CTR 0.29. Confidence: moderate. It could be wrong if the low CTR is mainly caused by its very low search position.

6. **content_0371fc7cdb0a** — Action: Improve CTR. Reason: CTR_FIX because the page is deep with CTR 0.04. Confidence: moderate. It could be wrong if the page is not relevant to valuable search queries.

7. **content_03b7a0f49f84** — Action: Improve CTR. Reason: CTR_FIX because the page is deep with CTR 0.00. Confidence: moderate. It could be wrong if the page needs broader SEO changes rather than a CTR-focused change.

8. **content_04e97796ac4d** — Action: Improve CTR. Reason: CTR_FIX because the page is deep with CTR 0.11. Confidence: moderate. It could be wrong if its search position is too low for a snippet change to matter.

9. **content_0527b708fdec** — Action: Improve CTR. Reason: CTR_FIX because the page is deep with CTR 0.00. Confidence: moderate. It could be wrong if the page has insufficient search visibility.

10. **content_07445a84d693** — Action: Improve CTR. Reason: CTR_FIX because the page is deep with CTR 0.00. Confidence: moderate. It could be wrong if the page's main problem is ranking rather than CTR.

11. **content_07f3e2bf4f9b** — Action: Improve CTR. Reason: CTR_FIX because the page is deep with CTR 0.00. Confidence: moderate. It could be wrong if the content does not match useful search intent.

12. **content_087348c25d8c** — Action: Improve CTR. Reason: CTR_FIX because the page is deep with CTR 0.00. Confidence: moderate. It could be wrong if the page needs content or ranking improvements first.

13. **content_08eb507fdfc2** — Action: Improve CTR. Reason: CTR_FIX because the page is deep with CTR 0.05. Confidence: moderate. It could be wrong if the low CTR is mainly a consequence of its position.

14. **content_08eeb3ea4955** — Action: Improve CTR. Reason: CTR_FIX because the page is deep with CTR 0.00. Confidence: moderate. It could be wrong if the page has little useful search demand.

15. **content_0ac4a4e97bd9** — Action: Improve CTR. Reason: CTR_FIX because the page is deep with CTR 0.00. Confidence: moderate. It could be wrong if the underlying ranking problem is more important than the snippet.

16. **content_0e57843e6749** — Action: Improve CTR. Reason: CTR_FIX because the page is deep with CTR 0.00. Confidence: moderate. It could be wrong if a CTR change would not affect its visibility.

17. **content_0e7efaeb2c67** — Action: Improve CTR. Reason: CTR_FIX because the page is deep with CTR 0.00. Confidence: moderate. It could be wrong if the page needs a larger content or SEO improvement.

18. **content_0e8a62788a4a** — Action: Improve CTR. Reason: CTR_FIX because the page is deep with CTR 0.16. Confidence: moderate. It could be wrong if the low CTR is mainly explained by its search position.

19. **content_0f16d0555526** — Action: Improve CTR. Reason: CTR_FIX because the page is deep with CTR 0.00. Confidence: moderate. It could be wrong if the page has low search demand or poor relevance.

20. **content_0f48c31cef7c** — Action: Improve CTR. Reason: CTR_FIX because the page is deep with CTR 0.00. Confidence: moderate. It could be wrong if improving the snippet would not address the main reason for its low ranking.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*



The baseline has a useful directional signal, but some picks may be weak because search position alone does not prove that a CTR change will improve performance. A page can rank deeply because of relevance, content quality, competition, or search demand, so a CTR-focused action may not always be the correct intervention.

The rule uses current dataset signals such as `position_tier`, `freshness_tier`, `ctr`, and `avg_position`. I did not use future-window performance or a target/label-derived field to calculate the score.

The baseline should therefore be treated as decision-support rather than proof that every selected page needs the recommended action.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show a few lower-scoring picks for a weak-pick review
weak_picks = queue.sort_values(
    ["score", "content_id"],
    ascending=[True, True]
).head(10)[
    [
        "content_id",
        "score",
        "reason_code",
        "action",
        "position_tier",
        "freshness_tier",
        "ctr",
        "avg_position"
    ]
]

print("Weak-pick candidates:")
print(weak_picks.to_string(index=False))


# Leakage check: verify that the scoring rule uses only the intended current signals
score_columns = ["position_tier", "freshness_tier"]

print("\nSignals used by the baseline score:")
print(score_columns)

print("\nNo future-window or label-derived inputs were used in the scoring formula.")


Weak-pick candidates:
          content_id  score     reason_code           action position_tier freshness_tier    ctr  avg_position
content_0008d03a5c6c    0.0 REFRESH_SUPPORT Review freshness         top_3           0-30   0.00           0.0
content_0015d80e7092    0.0 REFRESH_SUPPORT Review freshness         top_3           0-30   0.00           2.0
content_0022a6b4290f    0.0 REFRESH_SUPPORT Review freshness         top_3           0-30   0.07           1.2
content_004d8a5ce838    0.0 REFRESH_SUPPORT Review freshness         top_3           0-30   0.00           0.0
content_006b16e7a2e7    0.0 REFRESH_SUPPORT Review freshness         top_3           0-30 100.00           1.0
content_007365c3d77d    0.0 REFRESH_SUPPORT Review freshness         top_3           0-30   0.00           0.0
content_007f55873658    0.0 REFRESH_SUPPORT Review freshness         top_3           0-30   0.00           0.0
content_00ce7a27ff8b    0.0 REFRESH_SUPPORT Review freshness         top_3           0-30 

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [23]:
import os

path = "/content/flyrank-ml-internship/work/outputs/baseline_action_score.csv"

print("CSV exists:", os.path.exists(path))
print("CSV size:", os.path.getsize(path), "bytes")

CSV exists: True
CSV size: 1702690 bytes
